# DocPilot benchmark: plain RAG vs agentic RAG on SQuAD 2.0

Runs on a normal CPU runtime (no GPU needed). Add a Groq API key first: 🔑 Secrets → `GROQ_API_KEY`, notebook access on. Then Runtime → Run all. Takes about 20–60 minutes, depending on Groq API rate limits.

In [ ]:
!git clone -q https://github.com/khush-ahuja/DocPilot
%cd DocPilot
!pip -q install -r requirements.txt datasets

In [ ]:
import os
from google.colab import userdata
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

In [ ]:
!python eval/squad_bench.py --model qwen/qwen3.8-27b --articles 8 --answerable 36 --unanswerable 24

In [ ]:
import json
print(json.dumps(json.load(open('eval/squad_results.json')), indent=2))

In [ ]:
# where DocPilot and plain RAG disagree
rows = [json.loads(l) for l in open('eval/squad_samples.jsonl')]
by = {}
for r in rows: by.setdefault(r['id'], {})[r['system']] = r
for q in by.values():
    a, b = q['plain_rag'], q['docpilot']
    if a['label'] != b['label']:
        print('Q:', a['question'], '| gold:', a['gold'] or 'NONE')
        print('  plain   ', a['label'], '->', a['answer'][:150])
        print('  docpilot', b['label'], '->', b['answer'][:150], '\n')

In [ ]:
from google.colab import files
files.download('eval/squad_results.json'); files.download('eval/squad_samples.jsonl')